# Ensamble de experimentos (2 etapas)

**Etapa 1 (por experimento):** promedia las probabilidades de las 10 semillas de cada experimento. Si el archivo `prediccion_<experimento>-avg.txt` ya existe, lo usa directamente; si no, lo calcula a partir de `prediccion_<base>-<semilla>.txt` y lo graba.

**Etapa 2 (entre experimentos):** promedia los N resultados de la etapa 1, con el mismo peso para cada experimento (no para cada semilla individual).

**Etapa 3:** sube el ensamble final a Kaggle, mismos cortes que siempre.

## 1) Librerías

In [ ]:
suppressMessages({
  require("data.table")
})


## 2) Configuración

Un `carpeta` por experimento porque cada uno vivía en su propia carpeta de trabajo (`WF9200`, `WF9202`, etc.). `label` es el nombre tal cual aparece en el archivo de predicción ya promediado por semilla (`prediccion_<label>.txt`); `base_name` es ese mismo nombre sin el sufijo `-avg`, usado solo si hay que recalcular el promedio de semillas porque el archivo `-avg` no existe todavía.

In [ ]:
PARAM <- list()

# Las 10 semillas usadas en todos los experimentos (fallback, solo se usan si
# hay que recalcular el promedio por semilla de algun experimento)
PARAM$semillas <- c(468889, 567793, 347671, 678607, 702787, 117809, 925387, 382961, 744559, 777199)

PARAM$experimentos <- list(
  list(nombre = "estandarizar con bug",
       experimento = 9200,
       label = "CA-MachineLearning_DR-estandarizar_FEintra-SI_roll0-avg",
       base_name = "CA-MachineLearning_DR-estandarizar_FEintra-SI_roll0",
       carpeta = "/content/buckets/b1/exp/WF9200"),                        

  list(nombre = "estandarizar sin bug",
       experimento = 9202,
       label = "CA-MachineLearning_DR-estandarizar_FEintra-SI_roll0-avg",
       base_name = "CA-MachineLearning_DR-estandarizar_FEintra-SI_roll0",
       carpeta = "/content/buckets/b1/exp/WF9202"),                        

  list(nombre = "dolar blue",
       experimento = 9201,
       label = "CA-MachineLearning_DR-dolar_blue_FEintra-SI_roll0-avg",
       base_name = "CA-MachineLearning_DR-dolar_blue_FEintra-SI_roll0",
       carpeta = "/content/buckets/b1/exp/WF9201"),                        

  list(nombre = "deflacion (baseline)",
       experimento = 9105,
       label = "lags_deltas-avg",
       base_name = "lags_deltas",
       carpeta = "/content/buckets/b1/exp/WF9105"),                         

  list(nombre = "ninguno",
       experimento = 9203,
       label = "CA-MachineLearning_DR-ninguno_FEintra-SI_roll0-avg",
       base_name = "CA-MachineLearning_DR-ninguno_FEintra-SI_roll0",
       carpeta = "/content/buckets/b1/exp/WF9203"),                        

  list(nombre = "agregando rolling mean 3",
       experimento = 9106,
       label = "lags1-2_delta1-2_avg-3-avg",
       base_name = "lags1-2_delta1-2_avg-3",
       carpeta = "/content/buckets/b1/exp/WF9106")                         
)

# Configuracion del ensamble final / submit
PARAM$carpeta_salida       <- "/content/buckets/b1/exp/WF_ensemble_final"  # <-- AJUSTAR
PARAM$experimento_final    <- 9999                                         # <-- AJUSTAR
PARAM$experiment_name_final <- "ensemble_6experimentos-avg"

PARAM$kaggle <- list()
PARAM$kaggle$competencia <- "utn-2026-virtual-jr"
PARAM$kaggle$cortes <- seq(1800, 2400, by = 100)
PARAM$kaggle$delay_segundos <- 30


## 3) Etapa 1 — obtener (o calcular) el promedio de semillas de cada experimento

Para cada experimento: si existe `prediccion_<label>.txt` en su carpeta, lo lee directamente. Si no existe, arma el promedio leyendo `prediccion_<base_name>-<semilla>.txt` para las 10 semillas y lo graba con ese mismo nombre `-avg`, para no tener que recalcularlo la próxima vez.

In [ ]:
obtener_avg_experimento <- function(exp) {

  archivo_avg <- file.path(exp$carpeta, sprintf("prediccion_%s.txt", exp$label))

  if (file.exists(archivo_avg)) {
    cat(sprintf("[%s] uso el avg ya calculado: %s\n", exp$nombre, archivo_avg))
    tb <- fread(archivo_avg, sep = "\t")
    return(tb[, .(numero_de_cliente, prob)])
  }

  cat(sprintf("[%s] no encontre %s, lo calculo desde las %d semillas...\n",
    exp$nombre, archivo_avg, length(PARAM$semillas)))

  tablas <- list()
  for (semilla in PARAM$semillas) {
    archivo_semilla <- file.path(exp$carpeta,
      sprintf("prediccion_%s-%d.txt", exp$base_name, semilla))

    if (!file.exists(archivo_semilla)) {
      stop(sprintf("[%s] tampoco encontre: %s", exp$nombre, archivo_semilla))
    }

    tablas[[length(tablas) + 1]] <- fread(archivo_semilla, sep = "\t")
  }

  tb_avg <- rbindlist(tablas)[, .(prob = mean(prob)), by = numero_de_cliente]

  fwrite(tb_avg, file = archivo_avg, sep = "\t")
  cat(sprintf("[%s] guardado: %s\n", exp$nombre, archivo_avg))

  tb_avg
}


## 4) Ejecutar etapa 1 para los 6 experimentos

In [ ]:
predicciones_por_experimento <- list()

for (exp in PARAM$experimentos) {
  tb <- obtener_avg_experimento(exp)
  setnames(tb, "prob", exp$nombre)  # renombro para poder mergear despues
  predicciones_por_experimento[[exp$nombre]] <- tb
}


## 5) Etapa 2 — promedio entre experimentos

Hace un merge por `numero_de_cliente` de los N resultados y promedia esas N columnas de probabilidad, fila por fila. Si algún cliente falta en algún experimento, avisa (no debería pasar si todos corrieron sobre el mismo dataset de futuro).

In [ ]:
tb_final <- Reduce(function(a, b) merge(a, b, by = "numero_de_cliente", all = TRUE),
                    predicciones_por_experimento)

filas_incompletas <- sum(!complete.cases(tb_final))
if (filas_incompletas > 0) {
  cat(sprintf("AVISO: %d clientes no aparecen en los 6 experimentos. Se promedia con los que haya (na.rm=TRUE).\n",
    filas_incompletas))
}

columnas_prob <- names(tb_final)[-1] # todas menos numero_de_cliente
tb_final[, prob := rowMeans(.SD, na.rm = TRUE), .SDcols = columnas_prob]

tb_prediccion_ensemble <- tb_final[, .(numero_de_cliente, prob)]

dir.create(PARAM$carpeta_salida, showWarnings = FALSE, recursive = TRUE)
archivo_final <- file.path(PARAM$carpeta_salida,
  sprintf("prediccion_%s.txt", PARAM$experiment_name_final))
fwrite(tb_prediccion_ensemble, file = archivo_final, sep = "\t")

cat(sprintf("Ensamble final de %d experimentos guardado en: %s (%d clientes)\n",
  length(PARAM$experimentos), archivo_final, nrow(tb_prediccion_ensemble)))


## 6) Submit a Kaggle — mismo estilo que `kaggle_submit()` original

Mismos cortes, sin `shQuote` en `-f`, ruta relativa `./kaggle/...` respecto de `PARAM$carpeta_salida`.

In [ ]:
kaggle_submit_ensemble <- function(tb_prediccion, experimentos, archivo_final) {

  setwd(PARAM$carpeta_salida)
  setorder(tb_prediccion, -prob)
  dir.create("kaggle", showWarnings = FALSE)

  nombres_str <- paste(sapply(experimentos, function(e) e$nombre), collapse = " | ")

  for (envios in PARAM$kaggle$cortes) {

    tb_prediccion[, Predicted := 0L]
    tb_prediccion[1:envios, Predicted := 1L]

    archivo_kaggle <- sprintf(
        "./kaggle/KA%d_%s_%d.csv",
        PARAM$experimento_final,
        PARAM$experiment_name_final,
        envios
    )

    fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
      file = archivo_kaggle,
      sep = ","
    )

    kaggle <- file.path(Sys.getenv("HOME"), ".venv", "bin", "kaggle")
    comando <- paste(shQuote(kaggle), "competitions submit")
    competencia <- paste("-c", PARAM$kaggle$competencia)
    arch <- paste("-f", archivo_kaggle)

    mensaje <- sprintf(
        "-m 'experimento=%s envios=%d \n\nEnsamble de %d experimentos (mismo peso c/u): %s'",
        PARAM$experiment_name_final,
        envios,
        length(experimentos),
        nombres_str
    )

    linea <- paste(comando, competencia, arch, mensaje)

    salida <- system(linea, intern = TRUE)
    cat(salida, "\n")
    flush.console()
    Sys.sleep(PARAM$kaggle$delay_segundos)
  }
}


## 7) Ejecutar el submit
Esta es la única celda que efectivamente sube archivos a Kaggle.

In [ ]:
kaggle_submit_ensemble(tb_prediccion_ensemble, PARAM$experimentos, archivo_final)
